# Distributed xrdcp throughput measurement

Runs `warm_xcache` (distributed xrdcp via dask) wrapped in `roastcoffea.MetricsCollector` and `acquire_client` profiling so we can measure how fast we can pull files from xcache.


## Config


In [ ]:
AF = "coffeacasa-gateway"  # options: [coffeacasa-condor, coffeacasa-gateway, purdue-af-k8s, purdue-af-slurm]
n_workers = 800
AUTO_CLOSE_CLIENT = False


## Dependencies


In [ ]:
COFFEA_VERSION = "2026.4.0"
COFFEA_PIP = f"coffea=={COFFEA_VERSION}" if "git" not in COFFEA_VERSION else COFFEA_VERSION

! pip install $COFFEA_PIP ;

WORKER_DEPENDENCIES = [COFFEA_PIP, "roastcoffea==0.1.2"]


## Imports


In [ ]:
import copy
import sys
import time
from pathlib import Path

# Add src to path
repo_root = Path.cwd()
src_dir = repo_root / "src"
examples_dir = repo_root / "example_cms"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if str(examples_dir) not in sys.path:
    sys.path.insert(0, str(examples_dir))

import cloudpickle
import intccms
import example_cms
cloudpickle.register_pickle_by_value(intccms)
cloudpickle.register_pickle_by_value(example_cms)

from intccms.schema import Config, load_config_with_restricted_cli
from intccms.utils.output import OutputDirectoryManager
from intccms.utils.dask_client import acquire_client
from intccms.utils.tools import warm_xcache
from intccms.datasets import DatasetManager

from roastcoffea import MetricsCollector


## Configuration setup


In [ ]:
from example_cms.configs.configuration import config as original_config

config = copy.deepcopy(original_config)

# Warm every file by default; set to an integer for a small test.
config["datasets"]["max_files"] = None

# Per-notebook outputs (profiling/, measurements/ live under here).
config["general"]["output_dir"] = "example_cms/outputs_xrdcp_throughput/"

cli_args = []
full_config = load_config_with_restricted_cli(config, cli_args)
validated_config = Config(**full_config)

output_manager = OutputDirectoryManager(
    root_output_dir=validated_config.general.output_dir,
    cache_dir=validated_config.general.cache_dir,
    metadata_dir=validated_config.general.metadata_dir,
    skimmed_dir=validated_config.general.skimmed_dir,
)

dataset_manager = DatasetManager(validated_config.datasets)
print(f"Datasets: {dataset_manager.list_processes()}")


## Distributed xrdcp under MetricsCollector + dask profiling


In [ ]:
with acquire_client(
    AF,
    num_workers=n_workers,
    close_after=AUTO_CLOSE_CLIENT,
    pip_packages=WORKER_DEPENDENCIES,
    profile_output_dir=f"{output_manager.root_output_dir}/profiling/",
    profile_suffix="xrdcp_throughput",
) as (client, cluster):
    with MetricsCollector(
        client=client,
        track_workers=True,
        worker_tracking_interval=1.0,
    ) as collector:
        t0 = time.perf_counter()
        results, meta = warm_xcache(dataset_manager, client)
        t1 = time.perf_counter()

    collector.save_measurement(f"{output_manager.root_output_dir}/measurements")
    metrics = collector.get_metrics()
    tracking_data = collector.tracking_data

print(f"\nwall time: {t1 - t0:.1f}s")
print(f"files: {meta['n_files']}")
print(f"total: {meta['total_GB']:.1f} GB")
print(f"throughput (wall clock): {meta['total_Gbps']:.2f} Gbps")
print(f"throughput (per worker): {meta['per_worker_Gbps']:.2f} Gbps")


## roastcoffea tables


In [ ]:
from rich.console import Console
from roastcoffea.export.reporter import (
    format_throughput_table,
    format_resources_table,
    format_timing_table,
)

console = Console()
console.rule("Throughput")
console.print(format_throughput_table(metrics))
console.rule("Resources")
console.print(format_resources_table(metrics))
console.rule("Timing")
console.print(format_timing_table(metrics))
